# Full-Scale CLS Modeling for `order = 1`

This notebook builds a reproducible train/test-only classification workflow for the dynamic-pricing project.

Guardrails enforced here:
- only `cls_train_full.csv` and `cls_test.csv` are loaded in executed cells
- `cls_val.csv` is not touched during model building, model comparison, or threshold tuning
- `cls_train_full.csv` is used only for training
- `cls_test.csv` is used only for model comparison and threshold tuning
- the final holdout section is prepared but intentionally left locked

## 1. Setup

The notebook imports only the libraries needed for data loading, preprocessing, model training, scoring, and controlled backend selection.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

LIGHTGBM_AVAILABLE = False
LIGHTGBM_IMPORT_ERROR = None
LGBMClassifier = None
try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    LIGHTGBM_IMPORT_ERROR = str(exc)

CATBOOST_AVAILABLE = False
CATBOOST_IMPORT_ERROR = None
CatBoostClassifier = None
try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception as exc:
    CATBOOST_IMPORT_ERROR = str(exc)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 20)

print("Setup complete.")
print(f"LightGBM available: {LIGHTGBM_AVAILABLE}")
print(f"CatBoost available: {CATBOOST_AVAILABLE}")
if not LIGHTGBM_AVAILABLE:
    print(f"LightGBM import note: {LIGHTGBM_IMPORT_ERROR}")
if not CATBOOST_AVAILABLE:
    print(f"CatBoost import note: {CATBOOST_IMPORT_ERROR}")

Setup complete.
LightGBM available: False
CatBoost available: False
LightGBM import note: No module named 'lightgbm'
CatBoost import note: No module named 'catboost'


## 2. Configuration

The configuration block fixes file paths, target handling, leakage rules, feature candidates, thresholds, and the Orange baseline used for comparison.

In [2]:
BASE_DIR = Path(r"c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS")
DATA_DIR = BASE_DIR / "Daten"
TRAIN_PATH = DATA_DIR / "cls_train_full.csv"
TEST_PATH = DATA_DIR / "cls_test.csv"
VAL_PATH = DATA_DIR / "cls_val.csv"

DEFAULT_METRICS_PATH = BASE_DIR / "full_scale_cls_test_metrics_default.csv"
THRESHOLD_RESULTS_PATH = BASE_DIR / "full_scale_cls_threshold_tuning_results.csv"
BEST_MODELS_PATH = BASE_DIR / "full_scale_cls_best_models.csv"
TEST_PREDICTIONS_PATH = BASE_DIR / "full_scale_cls_test_predictions.csv"
FEATURE_IMPORTANCE_PATH = BASE_DIR / "full_scale_cls_feature_importance.csv"

TARGET_COLUMN = "order"
RANDOM_STATE = 42
THRESHOLDS = np.round(np.arange(5, 81) / 100, 2)
EXPECTED_THRESHOLD_COUNT = 76

LEAKAGE_COLUMNS = [
    "lineID",
    "revenue",
    "quantity",
    "q_raw",
    "quantity_class",
    "qty_suspicious",
    "click",
    "basket",
]

CATEGORICAL_CANDIDATES = [
    "salesIndex",
    "category_norm",
    "pharmForm_norm",
    "has_campaign",
    "pid_segment",
    "group12",
    "group34",
    "price_diff_bin",
    "discount_bin",
]

ORANGE_BASELINE = {
    "model": "Gradient Boosting (100k/100k sample)",
    "threshold": 0.22,
    "f1": 0.478436,
    "precision": 0.357169,
    "recall": 0.724377,
    "mcc": 0.267452,
}

assert len(THRESHOLDS) == EXPECTED_THRESHOLD_COUNT

print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")
print(f"Threshold count: {len(THRESHOLDS)}")
print(f"Threshold range: {THRESHOLDS[0]:.2f} to {THRESHOLDS[-1]:.2f}")

Train path: c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS\Daten\cls_train_full.csv
Test path: c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\Modellierung\CLS\Daten\cls_test.csv
Threshold count: 76
Threshold range: 0.05 to 0.80


## 3. Load Train and Test Data

Only the full training split and the test split are loaded in the executed workflow. The validation file remains untouched.

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

if TARGET_COLUMN not in train_df.columns or TARGET_COLUMN not in test_df.columns:
    raise KeyError(f"Target column '{TARGET_COLUMN}' must exist in both train and test")


def print_target_distribution(split_name: str, series: pd.Series) -> None:
    counts = series.value_counts(dropna=False).sort_index()
    order_rate = pd.to_numeric(series, errors="raise").mean() * 100
    print(f"{split_name} target distribution:")
    print(counts.to_string())
    print(f"{split_name} order rate (%): {order_rate:.4f}")


print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print("\nTrain columns:")
print(train_df.columns.tolist())
print("\nTest columns:")
print(test_df.columns.tolist())
print()
print_target_distribution("Train", train_df[TARGET_COLUMN])
print()
print_target_distribution("Test", test_df[TARGET_COLUMN])

Train shape: (1521260, 29)
Test shape: (349447, 29)

Train columns:
['day', 'day_7', 'day_14', 'day_30', 'adFlag', 'availability', 'price', 'competitorPrice', 'salesIndex', 'category_norm', 'pharmForm_norm', 'has_campaign', 'group12', 'group34', 'is_greater_discount', 'price_per_unit', 'price_diff_bin', 'discount_bin', 'pid_total_events', 'click_time', 'basket_time', 'order_time', 'group12_order', 'group34_order', 'pid_prob', 'availability_likelihood', 'day_7_likelihood', 'pid_segment', 'order']

Test columns:
['day', 'day_7', 'day_14', 'day_30', 'adFlag', 'availability', 'price', 'competitorPrice', 'salesIndex', 'category_norm', 'pharmForm_norm', 'has_campaign', 'group12', 'group34', 'is_greater_discount', 'price_per_unit', 'price_diff_bin', 'discount_bin', 'pid_total_events', 'click_time', 'basket_time', 'order_time', 'group12_order', 'group34_order', 'pid_prob', 'availability_likelihood', 'day_7_likelihood', 'pid_segment', 'order']

Train target distribution:
order
0    1182841
1   

## 4. Leakage Checks and Feature Contract

The workflow validates the target, removes forbidden leakage columns when present, and enforces an exact train/test feature match without silent assumptions.

In [4]:
y_train = pd.to_numeric(train_df[TARGET_COLUMN], errors="raise").astype(int)
y_test = pd.to_numeric(test_df[TARGET_COLUMN], errors="raise").astype(int)

for split_name, target_series in [("Train", y_train), ("Test", y_test)]:
    unique_values = set(target_series.unique().tolist())
    if unique_values != {0, 1}:
        raise ValueError(f"{split_name} target must contain only 0/1, found {sorted(unique_values)}")

raw_train_feature_columns = [column for column in train_df.columns if column != TARGET_COLUMN]
raw_test_feature_columns = [column for column in test_df.columns if column != TARGET_COLUMN]

if raw_train_feature_columns != raw_test_feature_columns:
    missing_in_test = [column for column in raw_train_feature_columns if column not in raw_test_feature_columns]
    additional_in_test = [column for column in raw_test_feature_columns if column not in raw_train_feature_columns]
    raise ValueError(
        "Train/Test feature mismatch detected. "
        f"Missing in test: {missing_in_test}. Additional in test: {additional_in_test}."
    )

present_leakage_columns = [column for column in LEAKAGE_COLUMNS if column in raw_train_feature_columns]
removed_leakage_columns = present_leakage_columns.copy()
already_absent_leakage_columns = [column for column in LEAKAGE_COLUMNS if column not in raw_train_feature_columns]

feature_columns = [column for column in raw_train_feature_columns if column not in LEAKAGE_COLUMNS]
X_train_raw = train_df[feature_columns].copy()
X_test_raw = test_df[feature_columns].copy()

if any(column in X_train_raw.columns for column in LEAKAGE_COLUMNS):
    raise ValueError("Leakage columns are still present in X_train_raw")
if any(column in X_test_raw.columns for column in LEAKAGE_COLUMNS):
    raise ValueError("Leakage columns are still present in X_test_raw")

print("Leakage columns present in exports:")
print(present_leakage_columns if present_leakage_columns else "None")
print("\nLeakage columns removed from modeling features:")
print(removed_leakage_columns if removed_leakage_columns else "None")
print("\nLeakage columns already absent:")
print(already_absent_leakage_columns if already_absent_leakage_columns else "None")
print(f"\nRaw feature count before leakage removal: {len(raw_train_feature_columns)}")
print(f"Feature count after leakage removal: {len(feature_columns)}")
print("Train/Test feature columns match exactly after target removal and leakage filtering.")

Leakage columns present in exports:
None

Leakage columns removed from modeling features:
None

Leakage columns already absent:
['lineID', 'revenue', 'quantity', 'q_raw', 'quantity_class', 'qty_suspicious', 'click', 'basket']

Raw feature count before leakage removal: 28
Feature count after leakage removal: 28
Train/Test feature columns match exactly after target removal and leakage filtering.


## 5. Feature Typing

Categorical features are assigned from the explicit project list when present. All remaining modeling features are treated as numeric, with `availability` kept numeric by design.

In [5]:
categorical_features = [
    column for column in CATEGORICAL_CANDIDATES
    if column in feature_columns and column != "availability"
]
numeric_features = [column for column in feature_columns if column not in categorical_features]

if "availability" in categorical_features:
    raise ValueError("availability must remain numeric/ordinal, not categorical")
if "availability" in feature_columns and "availability" not in numeric_features:
    raise ValueError("availability should be part of the numeric feature set")

print(f"Categorical feature count: {len(categorical_features)}")
print(categorical_features)
print(f"\nNumeric feature count: {len(numeric_features)}")
print(numeric_features)

Categorical feature count: 9
['salesIndex', 'category_norm', 'pharmForm_norm', 'has_campaign', 'pid_segment', 'group12', 'group34', 'price_diff_bin', 'discount_bin']

Numeric feature count: 19
['day', 'day_7', 'day_14', 'day_30', 'adFlag', 'availability', 'price', 'competitorPrice', 'is_greater_discount', 'price_per_unit', 'pid_total_events', 'click_time', 'basket_time', 'order_time', 'group12_order', 'group34_order', 'pid_prob', 'availability_likelihood', 'day_7_likelihood']


## 6. Helper Functions

The helper layer standardizes metric computation, threshold ranking, backend-specific data preparation, and feature-importance extraction.

In [6]:
def safe_logloss(y_true: pd.Series, y_prob: np.ndarray) -> float:
    clipped = np.clip(np.asarray(y_prob, dtype=float), 1e-15, 1 - 1e-15)
    return float(log_loss(y_true, clipped, labels=[0, 1]))


def compute_threshold_metrics(
    y_true: pd.Series,
    y_prob: np.ndarray,
    threshold: float,
    model_name: str,
    auc_value: float,
    logloss_value: float,
) -> dict:
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "threshold": float(threshold),
        "auc": float(auc_value),
        "logloss": float(logloss_value),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def evaluate_threshold_grid(
    y_true: pd.Series,
    y_prob: np.ndarray,
    model_name: str,
    auc_value: float,
    logloss_value: float,
    thresholds: np.ndarray,
) -> pd.DataFrame:
    rows = [
        compute_threshold_metrics(y_true, y_prob, threshold, model_name, auc_value, logloss_value)
        for threshold in thresholds
    ]
    return pd.DataFrame(rows)


def rank_by_f1(results: pd.DataFrame) -> pd.DataFrame:
    ranked = results.assign(threshold_distance=(results["threshold"] - 0.50).abs())
    ranked = ranked.sort_values(
        by=["f1", "mcc", "precision", "recall", "threshold_distance", "model"],
        ascending=[False, False, False, False, True, True],
    )
    return ranked.drop(columns=["threshold_distance"]).reset_index(drop=True)


def rank_by_mcc(results: pd.DataFrame) -> pd.DataFrame:
    ranked = results.assign(threshold_distance=(results["threshold"] - 0.50).abs())
    ranked = ranked.sort_values(
        by=["mcc", "f1", "precision", "recall", "threshold_distance", "model"],
        ascending=[False, False, False, False, True, True],
    )
    return ranked.drop(columns=["threshold_distance"]).reset_index(drop=True)


def select_best_threshold(model_results: pd.DataFrame, default_row: dict) -> tuple[dict, dict]:
    ranked = rank_by_f1(model_results)
    top_candidate = ranked.iloc[0].to_dict()
    default_precision = float(default_row["precision"])
    default_recall = float(default_row["recall"])
    precision_ratio = np.nan
    precision_collapse = False
    precision_message = "No precision collapse detected relative to the 0.50 baseline."

    if default_precision > 0:
        precision_ratio = float(top_candidate["precision"]) / default_precision
        precision_collapse = (
            float(top_candidate["precision"]) < 0.5 * default_precision
            and float(top_candidate["recall"]) > default_recall
        )

    if precision_collapse:
        plausible_mask = ~(
            (ranked["precision"] < 0.5 * default_precision)
            & (ranked["recall"] > default_recall)
        )
        plausible_candidates = ranked.loc[plausible_mask].reset_index(drop=True)
        if not plausible_candidates.empty:
            top_candidate = plausible_candidates.iloc[0].to_dict()
            precision_message = (
                "Top F1 candidate triggered the precision-collapse guard relative to 0.50; "
                "the best plausible alternative was selected."
            )
        else:
            precision_message = (
                "Top F1 candidate triggered the precision-collapse guard, but no alternative candidate satisfied the plausibility filter."
            )

    precision_check = {
        "precision_collapse_vs_050": bool(precision_collapse),
        "precision_ratio_vs_050": precision_ratio,
        "precision_check_message": precision_message,
    }
    return top_candidate, precision_check


def model_slug(model_name: str) -> str:
    slug = model_name.lower().replace(" ", "_").replace("-", "_")
    return "".join(character for character in slug if character.isalnum() or character == "_")


def prepare_lightgbm_frames(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    categorical_cols: list[str],
    numeric_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_prepared = train_frame.copy()
    test_prepared = test_frame.copy()

    for column in numeric_cols:
        train_prepared[column] = pd.to_numeric(train_prepared[column], errors="coerce")
        test_prepared[column] = pd.to_numeric(test_prepared[column], errors="coerce")

    for column in categorical_cols:
        train_series = train_frame[column].astype("string").fillna("__MISSING__")
        test_series = test_frame[column].astype("string").fillna("__MISSING__")
        train_categories = pd.Index(pd.unique(train_series))
        if "__UNSEEN__" not in train_categories:
            full_categories = train_categories.append(pd.Index(["__UNSEEN__"]))
        else:
            full_categories = train_categories
        train_prepared[column] = pd.Categorical(train_series, categories=full_categories)
        test_prepared[column] = pd.Categorical(
            test_series.where(test_series.isin(train_categories), "__UNSEEN__"),
            categories=full_categories,
        )

    return train_prepared, test_prepared


def prepare_catboost_frames(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    categorical_cols: list[str],
    numeric_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_prepared = train_frame.copy()
    test_prepared = test_frame.copy()

    for column in numeric_cols:
        train_prepared[column] = pd.to_numeric(train_prepared[column], errors="coerce")
        test_prepared[column] = pd.to_numeric(test_prepared[column], errors="coerce")

    for column in categorical_cols:
        train_prepared[column] = train_frame[column].astype("string").fillna("__MISSING__").astype(str)
        test_prepared[column] = test_frame[column].astype("string").fillna("__MISSING__").astype(str)

    return train_prepared, test_prepared


def prepare_histgb_frames(
    train_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    categorical_cols: list[str],
    numeric_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_prepared = pd.DataFrame(index=train_frame.index)
    test_prepared = pd.DataFrame(index=test_frame.index)

    for column in numeric_cols:
        train_prepared[column] = pd.to_numeric(train_frame[column], errors="coerce")
        test_prepared[column] = pd.to_numeric(test_frame[column], errors="coerce")

    for column in categorical_cols:
        train_series = train_frame[column].astype("string").fillna("__MISSING__")
        test_series = test_frame[column].astype("string").fillna("__MISSING__")
        categories = pd.Index(pd.unique(train_series))
        mapping = {category: index for index, category in enumerate(categories)}
        train_prepared[column] = train_series.map(mapping).astype(float)
        test_prepared[column] = test_series.map(mapping).fillna(-1).astype(float)

    return train_prepared[feature_columns], test_prepared[feature_columns]


def build_feature_importance_frame(model_name: str, estimator, feature_names: list[str]) -> pd.DataFrame | None:
    if model_name == "LightGBM" and hasattr(estimator, "feature_importances_"):
        importance_values = estimator.feature_importances_
    elif model_name == "CatBoost" and hasattr(estimator, "get_feature_importance"):
        importance_values = estimator.get_feature_importance()
    else:
        return None

    importance_df = pd.DataFrame(
        {
            "model": model_name,
            "feature": feature_names,
            "importance": importance_values,
        }
    ).sort_values(by="importance", ascending=False).reset_index(drop=True)
    return importance_df

## 7. Model Training

The notebook trains a Logistic Regression baseline and one tree-based main model selected by backend availability.

In [7]:
trained_models = []
model_status_rows = []
feature_importance_df = None
main_model_limitation_note = None

logistic_status = {
    "model": "Logistic Regression",
    "backend": "logistic_regression",
    "status": "not_started",
    "fit_seconds": np.nan,
    "error_message": "",
}

try:
    logistic_transformers = []
    if categorical_features:
        logistic_transformers.append(
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            )
        )
    if numeric_features:
        logistic_transformers.append(
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler(with_mean=False)),
                    ]
                ),
                numeric_features,
            )
        )

    logistic_preprocessor = ColumnTransformer(transformers=logistic_transformers, remainder="drop")
    logistic_pipeline = Pipeline(
        steps=[
            ("preprocessor", logistic_preprocessor),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    solver="saga",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    fit_start = time.perf_counter()
    logistic_pipeline.fit(X_train_raw, y_train)
    logistic_fit_seconds = time.perf_counter() - fit_start

    logistic_status.update(
        {
            "status": "success",
            "fit_seconds": logistic_fit_seconds,
            "error_message": "",
        }
    )
    trained_models.append(
        {
            "model": "Logistic Regression",
            "backend": "logistic_regression",
            "estimator": logistic_pipeline,
            "x_test_input": X_test_raw.copy(),
            "fit_seconds": logistic_fit_seconds,
            "feature_names": feature_columns.copy(),
            "supports_feature_importance": False,
            "slug": model_slug("Logistic Regression"),
        }
    )
    print(f"Logistic Regression trained successfully in {logistic_fit_seconds:.2f} seconds.")
except MemoryError as exc:
    logistic_status.update(
        {
            "status": "skipped_memory_error",
            "error_message": str(exc),
        }
    )
    print(f"Logistic Regression skipped due to MemoryError: {exc}")
except Exception as exc:
    logistic_status.update(
        {
            "status": "failed",
            "error_message": str(exc),
        }
    )
    print(f"Logistic Regression failed and was skipped: {exc}")

model_status_rows.append(logistic_status)

if LIGHTGBM_AVAILABLE:
    selected_main_backend = "lightgbm"
    selected_main_model = "LightGBM"
elif CATBOOST_AVAILABLE:
    selected_main_backend = "catboost"
    selected_main_model = "CatBoost"
else:
    selected_main_backend = "hist_gradient_boosting"
    selected_main_model = "HistGradientBoosting"

main_status = {
    "model": selected_main_model,
    "backend": selected_main_backend,
    "status": "not_started",
    "fit_seconds": np.nan,
    "error_message": "",
}

print(f"Selected main-model backend: {selected_main_model}")

try:
    if selected_main_backend == "lightgbm":
        X_train_main, X_test_main = prepare_lightgbm_frames(X_train_raw, X_test_raw, categorical_features, numeric_features)
        main_estimator = LGBMClassifier(
            objective="binary",
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        fit_start = time.perf_counter()
        if categorical_features:
            main_estimator.fit(X_train_main, y_train, categorical_feature=categorical_features)
        else:
            main_estimator.fit(X_train_main, y_train)
        main_fit_seconds = time.perf_counter() - fit_start
        feature_importance_df = build_feature_importance_frame("LightGBM", main_estimator, feature_columns)
    elif selected_main_backend == "catboost":
        X_train_main, X_test_main = prepare_catboost_frames(X_train_raw, X_test_raw, categorical_features, numeric_features)
        main_estimator = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            iterations=500,
            learning_rate=0.05,
            depth=6,
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=100,
        )
        fit_start = time.perf_counter()
        if categorical_features:
            main_estimator.fit(X_train_main, y_train, cat_features=categorical_features)
        else:
            main_estimator.fit(X_train_main, y_train)
        main_fit_seconds = time.perf_counter() - fit_start
        feature_importance_df = build_feature_importance_frame("CatBoost", main_estimator, feature_columns)
    else:
        X_train_main, X_test_main = prepare_histgb_frames(X_train_raw, X_test_raw, categorical_features, numeric_features)
        main_estimator = HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_depth=8,
            random_state=RANDOM_STATE,
        )
        fit_start = time.perf_counter()
        main_estimator.fit(X_train_main, y_train)
        main_fit_seconds = time.perf_counter() - fit_start
        main_model_limitation_note = (
            "LightGBM and CatBoost were unavailable. HistGradientBoostingClassifier was used as fallback without native categorical handling or native feature importance export."
        )

    main_status.update(
        {
            "status": "success",
            "fit_seconds": main_fit_seconds,
            "error_message": "",
        }
    )
    trained_models.append(
        {
            "model": selected_main_model,
            "backend": selected_main_backend,
            "estimator": main_estimator,
            "x_test_input": X_test_main,
            "fit_seconds": main_fit_seconds,
            "feature_names": feature_columns.copy(),
            "supports_feature_importance": feature_importance_df is not None,
            "slug": model_slug(selected_main_model),
        }
    )
    print(f"{selected_main_model} trained successfully in {main_fit_seconds:.2f} seconds.")
    if main_model_limitation_note is not None:
        print(main_model_limitation_note)
except Exception as exc:
    main_status.update(
        {
            "status": "failed",
            "error_message": str(exc),
        }
    )
    print(f"{selected_main_model} failed and was skipped: {exc}")

model_status_rows.append(main_status)
model_status_df = pd.DataFrame(model_status_rows)
print("\nModel training log:")
print(model_status_df.to_string(index=False))

if not trained_models:
    raise RuntimeError("No model trained successfully. Review the model training log above.")

c:\Users\karim\OneDrive - FHNW\Documents\Analytics Project Code\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression trained successfully in 2513.50 seconds.
Selected main-model backend: HistGradientBoosting
HistGradientBoosting trained successfully in 27.13 seconds.
LightGBM and CatBoost were unavailable. HistGradientBoostingClassifier was used as fallback without native categorical handling or native feature importance export.

Model training log:
               model                backend  status  fit_seconds error_message
 Logistic Regression    logistic_regression success  2513.501236              
HistGradientBoosting hist_gradient_boosting success    27.134214              


## 8. Evaluation on Test and Threshold Tuning

Every successful model is scored on `cls_test.csv` at the default threshold and across the full threshold grid. Results are saved immediately after computation.

In [8]:
default_metrics_rows = []
threshold_result_frames = []
best_model_rows = []
test_predictions_df = pd.DataFrame({TARGET_COLUMN: y_test.reset_index(drop=True)})

for model_entry in trained_models:
    model_name = model_entry["model"]
    estimator = model_entry["estimator"]
    x_test_input = model_entry["x_test_input"]
    fit_seconds = model_entry["fit_seconds"]
    backend = model_entry["backend"]

    y_prob = estimator.predict_proba(x_test_input)[:, 1]
    auc_value = float(roc_auc_score(y_test, y_prob))
    logloss_value = safe_logloss(y_test, y_prob)

    default_row = compute_threshold_metrics(
        y_true=y_test,
        y_prob=y_prob,
        threshold=0.50,
        model_name=model_name,
        auc_value=auc_value,
        logloss_value=logloss_value,
    )
    default_row["backend"] = backend
    default_row["fit_seconds"] = fit_seconds
    default_metrics_rows.append(default_row)

    model_threshold_df = evaluate_threshold_grid(
        y_true=y_test,
        y_prob=y_prob,
        model_name=model_name,
        auc_value=auc_value,
        logloss_value=logloss_value,
        thresholds=THRESHOLDS,
    )
    model_threshold_df["backend"] = backend
    model_threshold_df["fit_seconds"] = fit_seconds
    threshold_result_frames.append(model_threshold_df)

    best_row, precision_check = select_best_threshold(model_threshold_df, default_row)
    best_row["backend"] = backend
    best_row["fit_seconds"] = fit_seconds
    best_row.update(precision_check)
    best_model_rows.append(best_row)

    model_slug_name = model_entry["slug"]
    test_predictions_df[f"{model_slug_name}_p_order_1"] = y_prob
    test_predictions_df[f"{model_slug_name}_pred_default_050"] = (y_prob >= 0.50).astype(int)
    test_predictions_df[f"{model_slug_name}_pred_best_threshold"] = (y_prob >= float(best_row["threshold"])).astype(int)


default_metrics_df = pd.DataFrame(default_metrics_rows)
threshold_results_df = pd.concat(threshold_result_frames, ignore_index=True)
best_models_df = pd.DataFrame(best_model_rows)

winner_rank = best_models_df.sort_values(
    by=["f1", "mcc", "precision", "recall", "threshold", "model"],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)
best_test_model_row = winner_rank.iloc[0].copy()
best_models_df["is_recommended_winner"] = (
    best_models_df["model"].eq(best_test_model_row["model"])
    & np.isclose(best_models_df["threshold"], float(best_test_model_row["threshold"]))
)

DEFAULT_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
default_metrics_df.to_csv(DEFAULT_METRICS_PATH, index=False)
threshold_results_df.to_csv(THRESHOLD_RESULTS_PATH, index=False)
best_models_df.to_csv(BEST_MODELS_PATH, index=False)
test_predictions_df.to_csv(TEST_PREDICTIONS_PATH, index=False)

if feature_importance_df is not None:
    feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

top10_f1_df = rank_by_f1(threshold_results_df).head(10).reset_index(drop=True)
top10_mcc_df = rank_by_mcc(threshold_results_df).head(10).reset_index(drop=True)
default_threshold_rows_df = default_metrics_df.sort_values(by=["f1", "mcc"], ascending=[False, False]).reset_index(drop=True)

print("Default-threshold metrics (0.50):")
print(default_threshold_rows_df.to_string(index=False))

print("\nTop 10 by F1:")
print(top10_f1_df.to_string(index=False))

print("\nTop 10 by MCC:")
print(top10_mcc_df.to_string(index=False))

print("\nRecommended winner on TEST:")
print(pd.DataFrame([best_test_model_row]).to_string(index=False))

if feature_importance_df is not None:
    print("\nTop 30 feature importances:")
    print(feature_importance_df.head(30).to_string(index=False))
else:
    print("\nNo feature importance file was created because the selected backend does not support the requested native importance export in this run.")

orange_comparison_df = pd.DataFrame(
    [
        {
            "orange_baseline_model": ORANGE_BASELINE["model"],
            "orange_threshold": ORANGE_BASELINE["threshold"],
            "orange_f1": ORANGE_BASELINE["f1"],
            "orange_precision": ORANGE_BASELINE["precision"],
            "orange_recall": ORANGE_BASELINE["recall"],
            "orange_mcc": ORANGE_BASELINE["mcc"],
            "test_model": best_test_model_row["model"],
            "test_threshold": best_test_model_row["threshold"],
            "test_f1": best_test_model_row["f1"],
            "test_precision": best_test_model_row["precision"],
            "test_recall": best_test_model_row["recall"],
            "test_mcc": best_test_model_row["mcc"],
            "delta_f1_vs_orange": float(best_test_model_row["f1"]) - ORANGE_BASELINE["f1"],
            "delta_precision_vs_orange": float(best_test_model_row["precision"]) - ORANGE_BASELINE["precision"],
            "delta_recall_vs_orange": float(best_test_model_row["recall"]) - ORANGE_BASELINE["recall"],
            "delta_mcc_vs_orange": float(best_test_model_row["mcc"]) - ORANGE_BASELINE["mcc"],
        }
    ]
)

print("\nComparison to Orange baseline:")
print(orange_comparison_df.to_string(index=False))

print("\nSaved files:")
print(f"- {DEFAULT_METRICS_PATH}")
print(f"- {THRESHOLD_RESULTS_PATH}")
print(f"- {BEST_MODELS_PATH}")
print(f"- {TEST_PREDICTIONS_PATH}")
if feature_importance_df is not None:
    print(f"- {FEATURE_IMPORTANCE_PATH}")

Default-threshold metrics (0.50):
               model  threshold      auc  logloss  accuracy  precision   recall       f1      mcc     tn    fp    fn    tp                backend  fit_seconds
 Logistic Regression        0.5 0.709999 0.632933  0.644218   0.367731 0.672124 0.475376 0.264643 168792 96849 27478 56328    logistic_regression  2513.501236
HistGradientBoosting        0.5 0.719114 0.493439  0.763006   0.537716 0.084123 0.145485 0.137748 259580  6061 76756  7050 hist_gradient_boosting    27.134214

Top 10 by F1:
               model  threshold      auc  logloss  accuracy  precision   recall       f1      mcc     tn     fp    fn    tp                backend  fit_seconds
HistGradientBoosting       0.22 0.719114 0.493439  0.624049   0.360131 0.730735 0.482480 0.274325 156832 108809 22566 61240 hist_gradient_boosting    27.134214
HistGradientBoosting       0.23 0.719114 0.493439  0.634794   0.365347 0.709245 0.482268 0.274387 162388 103253 24367 59439 hist_gradient_boosting    27.1

## 9. Assertions

These assertions verify the key train/test-only contract, the threshold-grid size, the absence of leakage columns in modeling matrices, and the existence of saved outputs.

In [9]:
assert train_df is not None and test_df is not None
assert TARGET_COLUMN in train_df.columns and TARGET_COLUMN in test_df.columns
assert set(y_train.unique().tolist()) == {0, 1}
assert set(y_test.unique().tolist()) == {0, 1}
assert raw_train_feature_columns == raw_test_feature_columns
assert all(column not in X_train_raw.columns for column in LEAKAGE_COLUMNS)
assert all(column not in X_test_raw.columns for column in LEAKAGE_COLUMNS)
threshold_counts = threshold_results_df.groupby("model")["threshold"].nunique()
assert (threshold_counts == EXPECTED_THRESHOLD_COUNT).all(), threshold_counts.to_dict()
for required_output in [
    DEFAULT_METRICS_PATH,
    THRESHOLD_RESULTS_PATH,
    BEST_MODELS_PATH,
    TEST_PREDICTIONS_PATH,
]:
    assert required_output.exists(), f"Missing required output file: {required_output}"
if feature_importance_df is not None:
    assert FEATURE_IMPORTANCE_PATH.exists(), f"Expected feature importance file missing: {FEATURE_IMPORTANCE_PATH}"

print("All train/test-only assertions passed.")

All train/test-only assertions passed.


## 10. Final Holdout Validation

This section evaluates the fixed final CLS baseline exactly once on `cls_val.csv`.

Locked decisions:
- model: `HistGradientBoosting`
- threshold: `0.22`
- training data: `cls_train_full.csv` only
- model selection / tuning reference: `cls_test.csv` only
- final evaluation: `cls_val.csv` only

No further model selection, threshold tuning, feature changes, or challenger models are allowed in this section.

In [10]:
RUN_FINAL_VAL = True
ALLOW_FINAL_VAL_RERUN = False

FINAL_MODEL_NAME = "HistGradientBoosting"
FINAL_BACKEND = "hist_gradient_boosting"
FINAL_THRESHOLD = 0.22
FINAL_VAL_METRICS_PATH = BASE_DIR / "final_cls_val_metrics.csv"
FINAL_VAL_PREDICTIONS_PATH = BASE_DIR / "final_cls_val_predictions.csv"
FINAL_VAL_CONFUSION_MATRIX_PATH = BASE_DIR / "final_cls_val_confusion_matrix.csv"

FINAL_TEST_REFERENCE = {
    "model": "HistGradientBoosting",
    "threshold": 0.22,
    "auc": 0.719114,
    "logloss": 0.493439,
    "f1": 0.482480,
    "precision": 0.360131,
    "recall": 0.730735,
    "mcc": 0.274325,
    "tn": 156832,
    "fp": 108809,
    "fn": 22566,
    "tp": 61240,
}

HISTGB_FINAL_PARAMS = {
    "learning_rate": 0.05,
    "max_iter": 300,
    "max_depth": 8,
    "random_state": RANDOM_STATE,
}

if not RUN_FINAL_VAL:
    print("Final VAL is locked.")
else:
    protected_outputs = [
        FINAL_VAL_METRICS_PATH,
        FINAL_VAL_PREDICTIONS_PATH,
        FINAL_VAL_CONFUSION_MATRIX_PATH,
    ]
    if not ALLOW_FINAL_VAL_RERUN and any(path.exists() for path in protected_outputs):
        existing_outputs = [str(path) for path in protected_outputs if path.exists()]
        raise RuntimeError(
            "Final VAL outputs already exist. The holdout evaluation is protected against accidental reruns. "
            f"Existing outputs: {existing_outputs}"
        )

    if not BEST_MODELS_PATH.exists():
        raise FileNotFoundError(f"Missing TEST reference file: {BEST_MODELS_PATH}")

    test_reference_df = pd.read_csv(BEST_MODELS_PATH)
    reference_mask = (
        test_reference_df["model"].eq(FINAL_MODEL_NAME)
        & np.isclose(test_reference_df["threshold"].astype(float), FINAL_THRESHOLD)
    )
    if reference_mask.sum() != 1:
        raise AssertionError(
            "Expected exactly one fixed TEST reference row for HistGradientBoosting at threshold 0.22."
        )

    fixed_test_reference_row = test_reference_df.loc[reference_mask].iloc[0].copy()
    for metric_name in ["auc", "logloss", "f1", "precision", "recall", "mcc"]:
        expected_value = float(FINAL_TEST_REFERENCE[metric_name])
        actual_value = float(fixed_test_reference_row[metric_name])
        if not np.isclose(actual_value, expected_value, atol=1e-6):
            raise AssertionError(
                f"TEST reference mismatch for {metric_name}: expected {expected_value}, got {actual_value}"
            )
    for metric_name in ["tn", "fp", "fn", "tp"]:
        expected_value = int(FINAL_TEST_REFERENCE[metric_name])
        actual_value = int(fixed_test_reference_row[metric_name])
        if actual_value != expected_value:
            raise AssertionError(
                f"TEST reference mismatch for {metric_name}: expected {expected_value}, got {actual_value}"
            )

    val_df = pd.read_csv(VAL_PATH)
    if TARGET_COLUMN not in val_df.columns:
        raise KeyError(f"Target column '{TARGET_COLUMN}' must exist in VAL")

    y_val = pd.to_numeric(val_df[TARGET_COLUMN], errors="raise").astype(int)
    unique_val_targets = set(y_val.unique().tolist())
    if unique_val_targets != {0, 1}:
        raise ValueError(f"VAL target must contain only 0/1, found {sorted(unique_val_targets)}")

    raw_val_feature_columns = [column for column in val_df.columns if column != TARGET_COLUMN]
    if raw_train_feature_columns != raw_val_feature_columns:
        missing_in_val = [column for column in raw_train_feature_columns if column not in raw_val_feature_columns]
        additional_in_val = [column for column in raw_val_feature_columns if column not in raw_train_feature_columns]
        raise ValueError(
            "Train/VAL feature mismatch detected. "
            f"Missing in VAL: {missing_in_val}. Additional in VAL: {additional_in_val}."
        )

    present_val_leakage_columns = [column for column in LEAKAGE_COLUMNS if column in raw_val_feature_columns]
    X_val_raw = val_df[feature_columns].copy()
    if any(column in X_val_raw.columns for column in LEAKAGE_COLUMNS):
        raise ValueError("Leakage columns are still present in X_val_raw")

    reuse_existing_histgb = (
        "main_estimator" in globals()
        and isinstance(main_estimator, HistGradientBoostingClassifier)
        and globals().get("selected_main_backend") == FINAL_BACKEND
    )

    X_train_histgb_final, X_val_histgb = prepare_histgb_frames(
        X_train_raw,
        X_val_raw,
        categorical_features,
        numeric_features,
    )

    final_fit_seconds = np.nan
    if reuse_existing_histgb:
        final_estimator = main_estimator
        final_model_source = "reused_kernel_model"
    else:
        final_estimator = HistGradientBoostingClassifier(**HISTGB_FINAL_PARAMS)
        fit_start = time.perf_counter()
        final_estimator.fit(X_train_histgb_final, y_train)
        final_fit_seconds = time.perf_counter() - fit_start
        final_model_source = "retrained_same_pipeline"

    val_prob = final_estimator.predict_proba(X_val_histgb)[:, 1]
    val_auc = float(roc_auc_score(y_val, val_prob))
    val_logloss = safe_logloss(y_val, val_prob)
    val_metrics_row = compute_threshold_metrics(
        y_true=y_val,
        y_prob=val_prob,
        threshold=FINAL_THRESHOLD,
        model_name=FINAL_MODEL_NAME,
        auc_value=val_auc,
        logloss_value=val_logloss,
    )

    test_metrics_reference = {
        "auc": float(FINAL_TEST_REFERENCE["auc"]),
        "logloss": float(FINAL_TEST_REFERENCE["logloss"]),
        "f1": float(FINAL_TEST_REFERENCE["f1"]),
        "precision": float(FINAL_TEST_REFERENCE["precision"]),
        "recall": float(FINAL_TEST_REFERENCE["recall"]),
        "mcc": float(FINAL_TEST_REFERENCE["mcc"]),
    }
    delta_metrics = {
        "delta_auc_vs_test": float(val_metrics_row["auc"]) - test_metrics_reference["auc"],
        "delta_logloss_vs_test": float(val_metrics_row["logloss"]) - test_metrics_reference["logloss"],
        "delta_f1_vs_test": float(val_metrics_row["f1"]) - test_metrics_reference["f1"],
        "delta_precision_vs_test": float(val_metrics_row["precision"]) - test_metrics_reference["precision"],
        "delta_recall_vs_test": float(val_metrics_row["recall"]) - test_metrics_reference["recall"],
        "delta_mcc_vs_test": float(val_metrics_row["mcc"]) - test_metrics_reference["mcc"],
    }

    stable_generalization = (
        abs(delta_metrics["delta_auc_vs_test"]) <= 0.02
        and abs(delta_metrics["delta_f1_vs_test"]) <= 0.03
        and abs(delta_metrics["delta_mcc_vs_test"]) <= 0.03
        and delta_metrics["delta_logloss_vs_test"] <= 0.05
    )
    clearly_worse_than_test = (
        delta_metrics["delta_auc_vs_test"] < -0.03
        or delta_metrics["delta_f1_vs_test"] < -0.05
        or delta_metrics["delta_mcc_vs_test"] < -0.05
        or delta_metrics["delta_logloss_vs_test"] > 0.05
    )
    possible_overfit_or_drift = (
        clearly_worse_than_test
        or delta_metrics["delta_recall_vs_test"] < -0.05
        or delta_metrics["delta_precision_vs_test"] < -0.05
    )
    final_baseline_acceptable = not clearly_worse_than_test

    interpretation_lines = [
        f"Stable generalization: {'yes' if stable_generalization else 'no'}.",
        f"VAL clearly worse than TEST: {'yes' if clearly_worse_than_test else 'no'}.",
        (
            "There are hints of overfitting or temporal drift."
            if possible_overfit_or_drift
            else "No strong evidence of overfitting or temporal drift from the TEST-to-VAL comparison."
        ),
        (
            "HistGradientBoosting remains a defensible final CLS baseline."
            if final_baseline_acceptable
            else "HistGradientBoosting should be treated cautiously as final CLS baseline because VAL degraded materially versus TEST."
        ),
    ]

    final_metrics_df = pd.DataFrame([
        {
            "model": FINAL_MODEL_NAME,
            "backend": FINAL_BACKEND,
            "eval_scope": "val",
            "threshold": FINAL_THRESHOLD,
            "auc": val_metrics_row["auc"],
            "logloss": val_metrics_row["logloss"],
            "accuracy": val_metrics_row["accuracy"],
            "precision": val_metrics_row["precision"],
            "recall": val_metrics_row["recall"],
            "f1": val_metrics_row["f1"],
            "mcc": val_metrics_row["mcc"],
            "tn": val_metrics_row["tn"],
            "fp": val_metrics_row["fp"],
            "fn": val_metrics_row["fn"],
            "tp": val_metrics_row["tp"],
            "model_source": final_model_source,
            "fit_seconds_if_retrained": final_fit_seconds,
            "train_rows": int(len(train_df)),
            "val_rows": int(len(val_df)),
            "test_reference_auc": test_metrics_reference["auc"],
            "test_reference_logloss": test_metrics_reference["logloss"],
            "test_reference_f1": test_metrics_reference["f1"],
            "test_reference_precision": test_metrics_reference["precision"],
            "test_reference_recall": test_metrics_reference["recall"],
            "test_reference_mcc": test_metrics_reference["mcc"],
            **delta_metrics,
            "stable_generalization": stable_generalization,
            "val_clearly_worse_than_test": clearly_worse_than_test,
            "possible_overfit_or_drift": possible_overfit_or_drift,
            "final_baseline_acceptable": final_baseline_acceptable,
            "interpretation": " ".join(interpretation_lines),
        }
    ])

    final_predictions_df = pd.DataFrame(
        {
            "model": FINAL_MODEL_NAME,
            "backend": FINAL_BACKEND,
            "eval_scope": "val",
            "row_index": val_df.index.to_numpy(),
            TARGET_COLUMN: y_val.to_numpy(),
            "p_order_1": val_prob,
            "threshold": FINAL_THRESHOLD,
            "pred_threshold_022": (val_prob >= FINAL_THRESHOLD).astype(int),
        }
    )

    final_confusion_matrix_df = pd.DataFrame([
        {
            "model": FINAL_MODEL_NAME,
            "backend": FINAL_BACKEND,
            "eval_scope": "val",
            "threshold": FINAL_THRESHOLD,
            "tn": int(val_metrics_row["tn"]),
            "fp": int(val_metrics_row["fp"]),
            "fn": int(val_metrics_row["fn"]),
            "tp": int(val_metrics_row["tp"]),
        }
    ])

    final_metrics_df.to_csv(FINAL_VAL_METRICS_PATH, index=False)
    final_predictions_df.to_csv(FINAL_VAL_PREDICTIONS_PATH, index=False)
    final_confusion_matrix_df.to_csv(FINAL_VAL_CONFUSION_MATRIX_PATH, index=False)

    print("Validated fixed TEST reference row:")
    print(pd.DataFrame([fixed_test_reference_row]).to_string(index=False))

    print("\nVAL leakage check:")
    print(f"Present leakage columns in raw VAL export: {present_val_leakage_columns if present_val_leakage_columns else 'None'}")
    print(f"VAL feature count after leakage removal: {len(feature_columns)}")
    print("No leakage columns remain in X_val_raw.")

    print("\nFinal VAL metrics at threshold 0.22:")
    print(final_metrics_df[[
        "model",
        "backend",
        "eval_scope",
        "threshold",
        "auc",
        "logloss",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "mcc",
        "tn",
        "fp",
        "fn",
        "tp",
        "model_source",
        "fit_seconds_if_retrained",
    ]].to_string(index=False))

    print("\nTEST vs VAL deltas:")
    print(final_metrics_df[[
        "delta_auc_vs_test",
        "delta_logloss_vs_test",
        "delta_f1_vs_test",
        "delta_precision_vs_test",
        "delta_recall_vs_test",
        "delta_mcc_vs_test",
    ]].to_string(index=False))

    print("\nInterpretation:")
    for line in interpretation_lines:
        print(f"- {line}")

    print("\nSaved files:")
    print(f"- {FINAL_VAL_METRICS_PATH}")
    print(f"- {FINAL_VAL_PREDICTIONS_PATH}")
    print(f"- {FINAL_VAL_CONFUSION_MATRIX_PATH}")

Validated fixed TEST reference row:
               model  threshold      auc  logloss  accuracy  precision   recall      f1      mcc     tn     fp    fn    tp                backend  fit_seconds  precision_collapse_vs_050  precision_ratio_vs_050                                       precision_check_message  is_recommended_winner
HistGradientBoosting       0.22 0.719114 0.493439  0.624049   0.360131 0.730735 0.48248 0.274325 156832 108809 22566 61240 hist_gradient_boosting    27.134214                      False                0.669742 No precision collapse detected relative to the 0.50 baseline.                   True

VAL leakage check:
Present leakage columns in raw VAL export: None
VAL feature count after leakage removal: 28
No leakage columns remain in X_val_raw.

Final VAL metrics at threshold 0.22:
               model                backend eval_scope  threshold      auc  logloss  accuracy  precision   recall       f1      mcc     tn     fp    fn    tp        model_source  fit_s